# UNNAT — GPU harness

Runs the whole pipeline on a GPU runtime and produces the numbers that decide the demo plan:
throughput per backbone, a metric DSM with uncertainty, an ablation table, and the GPU-only tests.

**Runtime → Change runtime type → GPU** before running anything.

No data download is required: the harness generates a synthetic town with a known DSM,
so every metric below is computed against real ground truth. Swap in your own GeoTIFF at
the bottom of the setup cell when you have one.


## 1. What GPU did we get


In [ ]:
!nvidia-smi


## 2. Get the code

Either point `REPO_URL` at your remote, or upload a zip of the repo when prompted.


In [ ]:
import os, pathlib

REPO_URL = ''   # e.g. 'https://github.com/<you>/unnat.git'

if REPO_URL:
    !git clone -q $REPO_URL unnat_repo
    os.chdir('unnat_repo')
elif not pathlib.Path('unnat').is_dir():
    from google.colab import files
    print('Upload a zip of the UNNAT repo...')
    up = files.upload()
    name = next(iter(up))
    !unzip -q -o $name
    root = next(p for p in pathlib.Path('.').iterdir() if (p / 'unnat').is_dir())
    os.chdir(root)
print('cwd:', os.getcwd())
!ls


## 3. Dependencies

Colab already has torch with CUDA. Only the geospatial and model stacks are missing.


In [ ]:
!pip install -q rasterio transformers
!python -m unnat.cli doctor --load dav2-vits


## 4. Test data with known ground truth

A 2048 px synthetic town at 0.5 m GSD, with correct CRS, sun angles and real ray-marched shadows.
`scene_dsm.tif` is the exact surface that produced the image, so it is a genuine reference.


In [ ]:
!python -m unnat.cli synth --out data/scene.tif --size 2048
!python -m unnat.cli info data/scene.tif


## 5. Throughput sweep

Which backbone can you afford at full resolution, and what batch size fits in this card.


In [ ]:
!python -m unnat.cli bench --image data/scene.tif \
    --backbones dav2-vits,dav2-vitl --chips 518,1024 --batches 1,2,4 \
    --device cuda --json out/bench.json


## 6. Full pipeline

Ingest → depth → semantics → shadow → anchors → AGMC → bootstrap σ → DSM, validated
against the reference. `--dem sim:` simulates a Copernicus GLO-30 from the true terrain,
which is a development harness, not evidence.


In [ ]:
!python -m unnat.cli run data/scene.tif --out out/run \
    --backbone dav2-vits --device cuda --batch 0 \
    --dem sim:data/scene_dtm.tif --ref data/scene_dsm.tif \
    --bootstrap 24 --json out/run_summary.json


## 7. Look at what came out


In [ ]:
import matplotlib.pyplot as plt, rasterio, numpy as np

def show(ax, path, title, cmap='terrain', **kw):
    with rasterio.open(path) as ds:
        a = ds.read(1, masked=True)
    im = ax.imshow(a, cmap=cmap, **kw); ax.set_title(title, fontsize=10); ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
with rasterio.open('data/scene.tif') as ds:
    axes[0,0].imshow(np.moveaxis(ds.read([1,2,3]), 0, -1)); axes[0,0].set_title('RGB', fontsize=10); axes[0,0].axis('off')
show(axes[0,1], 'out/run/dsm.tif', 'predicted DSM (m)')
show(axes[0,2], 'data/scene_dsm.tif', 'reference DSM (m)')
show(axes[1,0], 'out/run/ndsm.tif', 'height above ground (m)', cmap='viridis', vmin=0, vmax=30)
show(axes[1,1], 'out/run/sigma.tif', 'uncertainty 1 sigma (m)', cmap='magma')
show(axes[1,2], 'out/run/error.tif', 'error vs reference (m)', cmap='RdBu_r', vmin=-15, vmax=15)
plt.tight_layout(); plt.show()


## 8. Ablation: which parts earn their place

One inference, every calibration variant. Same depth field for every row, so the
difference between rows is the thing being ablated and nothing else.


In [ ]:
!python -m unnat.cli ablate data/scene.tif --ref data/scene_dsm.tif \
    --dem sim:data/scene_dtm.tif --backbone dav2-vits --device cuda --batch 0 \
    --json out/ablation.json

from IPython.display import Markdown, display
display(Markdown(open('out/ablation.md').read()))


## 9. GPU-only tests

These check that the GPU path produces the *same* surface as the CPU path:
batching is a scheduling decision, and fp16 must not move the answer.


In [ ]:
!python -m pytest tests -q -m gpu -v -rs


## 10. Calibration honesty check

σ is only worth showing if it predicts the error. One-sigma coverage should land near 0.68.


In [ ]:
import json
s = json.load(open('out/run_summary.json'))
m = s['metrics']
print(f"MAE          {m['mae_m']:.2f} m")
print(f"RMSE         {m['rmse_m']:.2f} m")
print(f"bias         {m['bias_m']:+.2f} m")
print(f"1-sigma cov  {m.get('coverage_1s', float('nan')):.2f}   (target 0.68)")
print(f"ECE          {m.get('ece_m', float('nan')):.2f} m")
print(f"baseline MAE {s['baseline_metrics']['mae_m']:.2f} m  (global affine)")
print()
for k, v in s['timings_s'].items():
    print(f'  {k:<14}{v:>7.1f}s')


## 11. Download the results


In [ ]:
!zip -qr unnat_results.zip out/
from google.colab import files
files.download('unnat_results.zip')
